In [1]:
import pandas as pd
import os

In [2]:
seed = 20
N_train = 1000

assert N_train <= 1000
DIR_RAW = './dataset_raw'
DIR_SPLIT_OUT = './dataset_split'

dataset_name_dir = 'markov_32feat_11t5g_more_balanced_largeN'
dataset_name_out = 'markov_32feat_11t5g_more_balanced_largeN_censor'

In [3]:
df_features = pd.read_csv(os.path.join(DIR_RAW, dataset_name_dir,'df_features.csv'), index_col=0)
df_features.index = df_features.index.rename('subject')
df_features = df_features.rename(columns={i:f'feat{i.zfill(2)}' for i in df_features.columns if i != 'subj'})
df_features = df_features.reset_index()

df_state_history_sampled_max = pd.read_csv(os.path.join(DIR_RAW, dataset_name_dir,'df_state_history_sampled_max.csv'),  index_col=0)
df_state_history_sampled_max = df_state_history_sampled_max.rename(columns={'state':'g', 'state_max_by_time':'g_max_by_time', 'time':'t'})
MAX_TIME_ALL = 10

In [4]:
df_true_prob = pd.read_csv(os.path.join(DIR_RAW, dataset_name_dir,'df_surfs.csv'), index_col=0)
id_vars = ['subj', 'g']
df_true_prob_long = df_true_prob.melt(
    id_vars=id_vars,
    value_vars=[i for i in df_true_prob.columns if i not in id_vars],
    value_name='true_prob',
    var_name='time'
)
df_true_prob_long = df_true_prob_long.rename(columns={'subj':'subject', 'time':'t'})
df_true_prob_long['t'] = df_true_prob_long['t'].astype(int)


In [5]:
import numpy as np
def random_censor_subj(df_subj, random_prop):
    n_tims = df_subj['t'].nunique()
    n_tims_keep = max(int(random_prop*1.2 * n_tims), 2) 
    out = df_subj.sort_values('t').iloc[:n_tims_keep,:]
    assert out.shape[0] > 1

    return out

def random_censor_obs(df_state_history_sampled_max_sub):
    np.random.seed(seed)
    n_subj = df_state_history_sampled_max_sub['subject'].nunique()
    prop_to_censor = np.random.rand(n_subj)
    out = []
    for idx, (subj, df_subj) in enumerate(
        df_state_history_sampled_max_sub.groupby('subject')
    ):
        out.append(random_censor_subj(df_subj, prop_to_censor[idx]))
    return pd.concat(out)


In [6]:
subj_train_all = df_features['subject'][::df_features['subject'].size//1000]
subj_train = subj_train_all.sample(n=N_train, random_state=seed)
subj_train_tune = subj_train.sample(n=min(N_train, 500), random_state=seed)
subj_val = df_features['subject'].loc[~df_features['subject'].isin(subj_train_all)].sample(n=500, random_state=seed)
subj_test = df_features['subject'].loc[
    ~(
        df_features['subject'].isin(subj_train_all) |
        df_features['subject'].isin(subj_val)
    )
]

for suffix, subj in [
    ('tune', subj_train_tune),
    ('train', subj_train),
    ('val', subj_val),
    ('test', subj_test)
]:
    print(f'n_subj in {suffix}:{len(subj)}')
    df_features_sub = df_features.loc[
        df_features['subject'].isin(subj),
        :
    ]
    df_features_sub.to_csv(os.path.join(DIR_SPLIT_OUT, f'{dataset_name_out}__df_features_{suffix}.csv'))

    df_state_history_sampled_max_sub = df_state_history_sampled_max.loc[
        df_state_history_sampled_max['subject'].isin(subj),
        :
    ]
    df_state_history_sampled_max_sub = random_censor_obs(df_state_history_sampled_max_sub)
    df_state_history_sampled_max_sub.to_csv(os.path.join(DIR_SPLIT_OUT, f'{dataset_name_out}__df_state_history_sampled_max_{suffix}.csv'))

    df_true_prob_long_sub = df_true_prob_long.loc[
        df_true_prob_long['subject'].isin(subj),
        :
    ]
    df_true_prob_long_sub.to_csv(os.path.join(DIR_SPLIT_OUT, f'{dataset_name_out}__df_true_prob_long_{suffix}.csv'))

del df_state_history_sampled_max_sub
del df_features_sub
del df_state_history_sampled_max
del df_features


n_subj in tune:500
n_subj in train:1000
n_subj in val:500
n_subj in test:2500
